# Tutorial 5: Evaluation

## Posterior Quality, Calibration, and Robustness

---

This notebook covers how to assess whether the learned posterior is trustworthy. Because our forward model is fully Gaussian, we can compute the **exact posterior** analytically and compare it directly to the neural approximation.

### What you'll learn

1. How to load a trained checkpoint and generate posterior predictions
2. The exact Bayesian posterior via the Woodbury identity
3. Hellinger distance — a metric for 2-D posterior agreement
4. P-P calibration plots and KS statistics
5. Robustness evaluation under structured masking

### Modules covered

| Module | Purpose |
|--------|---------|
| `src.evaluate` | Full evaluation pipeline |
| `src.exact_posterior` | Analytic posterior computation |
| `src.metrics` | Hellinger, KS, calibration, point error |
| `src.plots` | Visualisation utilities |

In [ ]:
import sys, os

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

import numpy as np
import torch
import matplotlib.pyplot as plt
import yaml

%matplotlib inline
plt.rcParams.update({'figure.dpi': 120, 'font.size': 11})

SEED = 42
torch.manual_seed(SEED)

---
## 1. Loading a Trained Model

Checkpoints store everything needed to reconstruct the model:
- `model_state_dict` — learned weights
- `config` — full YAML config used for training
- `model_type` — `'transformer'` or `'lstm'`

In [ ]:
from src.evaluate import load_model
from src.priors import FactorizedPrior
from src.dataset import FixedPulsarDataset

# Load config
cfg_path = os.path.join(PROJECT_ROOT, 'configs', 'transformer_v5.yaml')
with open(cfg_path) as f:
    cfg = yaml.safe_load(f)

device = torch.device('cpu')

# Try to load trained model
ckpt_path = os.path.join(PROJECT_ROOT, 'outputs', 'v5', 'transformer', 'best_model.pt')
has_checkpoint = os.path.exists(ckpt_path)

if has_checkpoint:
    model, model_type = load_model(ckpt_path, cfg, device)
    print(f'Loaded trained {model_type} from {ckpt_path}')
    n_params = sum(p.numel() for p in model.parameters())
    print(f'Parameters: {n_params:,}')
else:
    # Create untrained model for demonstration
    from src.models.model_wrappers import build_model
    model = build_model('transformer', cfg)
    model.eval()
    model_type = 'transformer'
    print('No checkpoint found — using untrained model for demonstration.')
    print('To get meaningful results, first run:')
    print('  python -m src.train --config configs/transformer_v5.yaml --model transformer')

In [ ]:
# Generate test data using the v5 factorized prior
prior = FactorizedPrior(cfg['prior']['global'], cfg['prior']['white_noise'], n_backends_max=4)
test_ds = FixedPulsarDataset(
    n_samples=10,
    prior=prior,
    data_cfg=cfg['data'],
    seed=cfg['training']['seed'] + 2_000_000,
    factorized=True,
)

print(f'Test dataset: {len(test_ds)} pulsars')
for i in range(3):
    item = test_ds[i]
    print(f'  [{i}] θ_global={item["theta_global"].numpy()}, N={item["seq_len"]}, '
          f'T={item["tspan_yr"]:.1f} yr')

---
## 2. Posterior Inference

For a single test pulsar, we:
1. Build a batch dict from the test item
2. Generate posterior samples via `model.sample_posterior()`
3. Evaluate log-probability on a 2-D grid via `model.log_prob_on_grid()`

In [ ]:
from src.models.tokenization import tokenize

def make_batch(item, device='cpu'):
    """Build a single-example batch from a factorized FixedPulsarDataset item."""
    sim = item['sim']
    tokens = tokenize(sim.t, sim.sigma, sim.residuals, sim.freq_mhz, sim.backend_id)
    L = len(sim.t)
    feat_keys = ['t_norm', 'dt_prev', 'r_over_sig', 'log_sigma', 'r_raw', 'freq_norm']
    features = torch.stack([tokens[k] for k in feat_keys], dim=-1).unsqueeze(0)
    return {
        'features': features.to(device),
        'backend_id': tokens['backend_id'].unsqueeze(0).to(device),
        'mask': torch.ones(1, L, dtype=torch.bool).to(device),
        'tspan_yr': torch.tensor([float(sim.t.max() - sim.t.min())]).to(device),
    }

# Pick test example
test_idx = 0
item = test_ds[test_idx]
batch = make_batch(item, device=device)

# Generate posterior samples
# FactorizedNPEModel.sample_posterior() returns (global_samples, wn_samples)
n_samples = 10000
with torch.no_grad():
    global_samples, wn_samples = model.sample_posterior(batch, n_samples=n_samples)

# global_samples: (B=1, n_samples, 4) — log10_A_red, gamma_red, log10_A_dm, gamma_dm
g_np = global_samples[0].numpy()  # (n_samples, 4)
true_theta = item['theta_global'].numpy()  # (4,)
global_names = ['log10_A_red', 'gamma_red', 'log10_A_dm', 'gamma_dm']

print(f'True θ_global:     [{", ".join(f"{v:.3f}" for v in true_theta)}]')
print(f'Posterior mean:    [{", ".join(f"{g_np[:, i].mean():.3f}" for i in range(4))}]')
print(f'Posterior std:     [{", ".join(f"{g_np[:, i].std():.3f}" for i in range(4))}]')
print(f'\nWN samples shape: {wn_samples.shape}  (B, n_backends_max, n_samples, 3)')

In [ ]:
# Evaluate the global flow on a 2D (log10_A_red, gamma_red) grid
# prior_bounds is set to the global prior for downstream cells
prior_bounds = cfg['prior']['global']
n_grid = 100
A_grid = torch.linspace(prior_bounds['log10_A_red'][0], prior_bounds['log10_A_red'][1], n_grid)
G_grid = torch.linspace(prior_bounds['gamma_red'][0], prior_bounds['gamma_red'][1], n_grid)
AA, GG = torch.meshgrid(A_grid, G_grid, indexing='ij')
grid_pts_2d = torch.stack([AA.reshape(-1), GG.reshape(-1)], dim=-1)

# Fix A_dm and gamma_dm at prior midpoints for the 2D visualisation slice
A_dm_mid = (prior_bounds['log10_A_dm'][0] + prior_bounds['log10_A_dm'][1]) / 2.0
gamma_dm_mid = (prior_bounds['gamma_dm'][0] + prior_bounds['gamma_dm'][1]) / 2.0
dm_cols = torch.tensor([[A_dm_mid, gamma_dm_mid]]).expand(n_grid * n_grid, -1)
grid_4d = torch.cat([grid_pts_2d, dm_cols], dim=-1).to(device)

with torch.no_grad():
    global_ctx, _ = model._get_contexts(batch)
    global_ctx_exp = global_ctx.expand(n_grid * n_grid, -1)
    theta_norm = model._normalize_global(grid_4d)
    log_probs_norm = model.global_flow.log_prob(theta_norm.float(), global_ctx_exp.float())
    log_probs = log_probs_norm - model.global_theta_std.log().sum()

log_probs_2d = log_probs.cpu().numpy().reshape(n_grid, n_grid)

# Normalise to density
dA = float(A_grid[1] - A_grid[0])
dG = float(G_grid[1] - G_grid[0])
log_max = np.max(log_probs_2d)
log_norm = log_max + np.log(np.sum(np.exp(log_probs_2d - log_max)) * dA * dG)
learned_post = np.exp(log_probs_2d - log_norm)

fig, ax = plt.subplots(figsize=(7, 5.5))
c = ax.contourf(A_grid.numpy(), G_grid.numpy(), learned_post.T, levels=30, cmap='Blues')
ax.plot(true_theta[0], true_theta[1], 'r*', ms=15, label='True θ')
plt.colorbar(c, ax=ax, label='Marginal density (A_dm, γ_dm at prior midpoints)')
ax.set_xlabel('$\\log_{10} A_{\\rm red}$')
ax.set_ylabel('$\\gamma_{\\rm red}$')
ax.set_title(f'Learned Global Posterior — red-noise slice (test example {test_idx})', fontsize=12)
ax.legend(fontsize=10)
fig.tight_layout()
plt.show()

---
## 3. Exact Bayesian Posterior

Because the signal model is **linear in Fourier coefficients** and the noise is **Gaussian**, the marginal likelihood has a closed-form expression. The data covariance is:

$$
\mathbf{C}(\theta) = \text{diag}(\boldsymbol{\sigma}^2 + j) + \mathbf{F} \boldsymbol{\Phi}(\theta) \mathbf{F}^\top
$$

The **Woodbury identity** enables efficient evaluation:
$$
\mathbf{C}^{-1} = \mathbf{D}^{-1} - \mathbf{D}^{-1}\mathbf{F}(\boldsymbol{\Phi}^{-1} + \mathbf{F}^\top\mathbf{D}^{-1}\mathbf{F})^{-1}\mathbf{F}^\top\mathbf{D}^{-1}
$$

This reduces the $\mathcal{O}(N^3)$ Cholesky to $\mathcal{O}(K^3)$ where $K = 2 \times$ n_modes $\ll N$.

In [ ]:
from src.exact_posterior import exact_posterior_grid

sim = item['sim']
jitter = cfg['data'].get('jitter', 1e-20)

exact = exact_posterior_grid(
    sim.residuals,
    sim.sigma,
    sim.F,
    sim.tspan,
    sim.n_modes,
    prior_bounds,
    n_grid=n_grid,
    jitter=jitter,
)

print('Exact posterior computed.')
print(f'Grid: {exact["log10_A_grid"].shape[0]} × {exact["gamma_grid"].shape[0]}')
print(f'MAP estimate: log10_A={exact["map_theta"][0]:.3f}, gamma={exact["map_theta"][1]:.3f}')
print(f'True values:  log10_A={true_theta[0]:.3f}, gamma={true_theta[1]:.3f}')

In [ ]:
# Side-by-side comparison: exact vs learned
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Exact posterior
c0 = axes[0].contourf(exact['log10_A_grid'], exact['gamma_grid'],
                       exact['posterior'].T, levels=30, cmap='Oranges')
axes[0].plot(true_theta[0], true_theta[1], 'r*', ms=15)
axes[0].set_title('Exact Posterior', fontsize=11)
plt.colorbar(c0, ax=axes[0])

# Learned posterior
c1 = axes[1].contourf(A_grid.numpy(), G_grid.numpy(),
                       learned_post.T, levels=30, cmap='Blues')
axes[1].plot(true_theta[0], true_theta[1], 'r*', ms=15)
axes[1].set_title('Learned Posterior', fontsize=11)
plt.colorbar(c1, ax=axes[1])

# Overlay
axes[2].contour(exact['log10_A_grid'], exact['gamma_grid'],
                exact['posterior'].T, levels=5, colors='orange', linewidths=1.5)
axes[2].contour(A_grid.numpy(), G_grid.numpy(),
                learned_post.T, levels=5, colors='steelblue', linewidths=1.5)
axes[2].plot(true_theta[0], true_theta[1], 'r*', ms=15)
axes[2].set_title('Overlay (orange=exact, blue=learned)', fontsize=11)

for ax in axes:
    ax.set_xlabel('$\\log_{10} A_{\\rm red}$')
    ax.set_ylabel('$\\gamma_{\\rm red}$')

fig.suptitle(f'Posterior Comparison — test example {test_idx}', fontsize=13, y=1.02)
fig.tight_layout()
plt.show()

---
## 4. Hellinger Distance

The **Hellinger distance** quantifies the agreement between two probability distributions:

$$
H(P, Q) = \frac{1}{\sqrt{2}} \sqrt{\sum_k \left(\sqrt{p_k} - \sqrt{q_k}\right)^2}
$$

- $H = 0$: distributions are identical
- $H = 1$: distributions have no overlap

Typical well-trained values: $H < 0.1$ (good), $H < 0.05$ (excellent).

In [ ]:
from src.metrics import hellinger_distance_grid

h = hellinger_distance_grid(exact['posterior'], learned_post)
print(f'Hellinger distance: {h:.4f}')

if has_checkpoint:
    print(f'  (With a trained model, expect H < 0.1)')
else:
    print(f'  (Untrained model — this will be high)')

In [ ]:
# Compute Hellinger for several test examples
hellinger_dists = []
n_eval = min(10, len(test_ds))

for i in range(n_eval):
    it = test_ds[i]
    sim_i = it['sim']
    batch_i = make_batch(it, device=device)
    
    # Exact posterior (2D red-noise marginal; raw sigma used as noise proxy)
    ex = exact_posterior_grid(
        sim_i.residuals, sim_i.sigma, sim_i.F, sim_i.tspan,
        sim_i.n_modes, prior_bounds, n_grid=n_grid, jitter=jitter)
    
    # Learned global posterior: 2D slice at prior midpoints for A_dm, gamma_dm
    with torch.no_grad():
        g_ctx, _ = model._get_contexts(batch_i)
        g_ctx_exp = g_ctx.expand(n_grid * n_grid, -1)
        theta_norm_i = model._normalize_global(grid_4d)
        lp_norm = model.global_flow.log_prob(theta_norm_i.float(), g_ctx_exp.float())
        lp = (lp_norm - model.global_theta_std.log().sum()).cpu().numpy()
    
    lp_2d = lp.reshape(n_grid, n_grid)
    lmax = np.max(lp_2d)
    lnorm = lmax + np.log(np.sum(np.exp(lp_2d - lmax)) * dA * dG)
    lpost = np.exp(lp_2d - lnorm)
    
    h_i = hellinger_distance_grid(ex['posterior'], lpost)
    hellinger_dists.append(h_i)
    print(f'  [{i}] θ_global=[{", ".join(f"{v:.2f}" for v in it["theta_global"][:2])}]  H={h_i:.4f}')

print(f'\nMean Hellinger: {np.mean(hellinger_dists):.4f} ± {np.std(hellinger_dists):.4f}')
print('(Comparison uses the red-noise 2D marginal; exact posterior marginalises over raw σ)')

---
## 5. P-P Calibration Plot

A **probability-probability (P-P)** plot checks whether the posterior is **well-calibrated**: if the true parameter lies at the $p$-th percentile of the marginal posterior, this should happen $p$% of the time. 

For each test example and parameter dimension:
1. Draw posterior samples → compute percentile rank of true value
2. Aggregate percentiles across all test examples
3. Plot empirical CDF vs diagonal

A well-calibrated model produces a line close to the diagonal.

In [ ]:
from src.metrics import calibration_percentiles, ks_statistic

# Collect global posterior samples for all test examples
all_true = []
all_samples = []

for i in range(n_eval):
    it = test_ds[i]
    batch_i = make_batch(it, device=device)
    with torch.no_grad():
        g_samps, _ = model.sample_posterior(batch_i, n_samples=5000)
    all_true.append(it['theta_global'].numpy())   # (4,)
    all_samples.append(g_samps[0].numpy())         # (5000, 4)

true_arr = np.array(all_true)      # (N, 4)
samples_arr = np.array(all_samples)  # (N, 5000, 4)

# Compute percentiles over all 4 global parameters
percentiles = calibration_percentiles(true_arr, samples_arr)  # (N, 4)

param_names = ['log10_A_red', 'gamma_red', 'log10_A_dm', 'gamma_dm']
ks_stats = [ks_statistic(percentiles[:, d]) for d in range(4)]

print('Calibration percentile summary:')
for d, name in enumerate(param_names):
    print(f'  {name}: mean={percentiles[:, d].mean():.3f}, '
          f'std={percentiles[:, d].std():.3f}, KS={ks_stats[d]:.4f}')
print(f'  Mean KS: {np.mean(ks_stats):.4f}')

In [ ]:
# P-P plot
fig, ax = plt.subplots(figsize=(6, 6))

expected = np.linspace(0, 1, 100)
ax.plot(expected, expected, 'k--', lw=1, label='Perfect calibration')

# 2σ envelope (for finite sample size)
n = len(percentiles)
sigma_band = 1.36 / np.sqrt(n)  # KS critical value at 95%
ax.fill_between(expected, expected - sigma_band, expected + sigma_band,
                alpha=0.15, color='gray', label=f'95% band (n={n})')

colors = ['C0', 'C1']
for d, (name, ks, color) in enumerate(zip(param_names, ks_stats, colors)):
    sorted_p = np.sort(percentiles[:, d])
    ecdf = np.arange(1, n + 1) / n
    ax.step(sorted_p, ecdf, where='post', lw=1.5, color=color,
            label=f'{name} (KS={ks:.3f})')

ax.set_xlabel('Posterior percentile rank')
ax.set_ylabel('Empirical CDF')
ax.set_title('P-P Calibration Plot', fontsize=12)
ax.legend(fontsize=9)
ax.set_xlim(0, 1); ax.set_ylim(0, 1)
ax.set_aspect('equal')
fig.tight_layout()
plt.show()

print('Interpretation:')
print('  Line above diagonal → posteriors too narrow (overconfident)')
print('  Line below diagonal → posteriors too broad (underconfident)')
print('  Line on diagonal → perfectly calibrated')

---
## 6. Point Estimate Error

In [ ]:
from src.metrics import point_estimate_error

# Posterior mean as point estimate (global params only)
means_arr = samples_arr.mean(axis=1)  # (N, 4)
pe = point_estimate_error(true_arr, means_arr)  # (N, 4)

fig, axes = plt.subplots(2, 2, figsize=(11, 8))
axes = axes.ravel()

for d, (name, ax) in enumerate(zip(param_names, axes)):
    ax.scatter(true_arr[:, d], means_arr[:, d], s=30, alpha=0.7)
    lims = [prior_bounds[name][0], prior_bounds[name][1]]
    ax.plot(lims, lims, 'k--', lw=1, label='Perfect')
    ax.set_xlabel(f'True {name}')
    ax.set_ylabel(f'Estimated {name}')
    ax.set_title(f'{name}  (MAE={pe[:, d].mean():.3f})', fontsize=10)
    ax.legend(fontsize=9)

fig.suptitle('Point Estimate Accuracy — global parameters (posterior mean)', fontsize=12, y=1.02)
fig.tight_layout()
plt.show()

---
## 7. Robustness Under Masking

A key advantage of the training augmentation strategy: the model should remain accurate even when test data are degraded by structured masking. We evaluate at several masking severity levels.

In [ ]:
from src.masking import apply_random_masking

masking_levels = [0.0, 0.15, 0.3, 0.45, 0.6]
n_robust = min(5, len(test_ds))

hellinger_by_level = {}

for sev in masking_levels:
    h_list = []
    for i in range(n_robust):
        it = test_ds[i]
        sim_i = it['sim']
        
        # Apply masking to tokens only (exact posterior always uses full data)
        rng = np.random.default_rng(999 + i)
        keep = apply_random_masking(sim_i.t, rng, severity=sev) if sev > 0 else None
        
        # Build masked batch manually
        t = sim_i.t[keep] if keep is not None else sim_i.t
        sigma = sim_i.sigma[keep] if keep is not None else sim_i.sigma
        r = sim_i.residuals[keep] if keep is not None else sim_i.residuals
        freq = sim_i.freq_mhz[keep] if keep is not None else sim_i.freq_mhz
        be = sim_i.backend_id[keep] if keep is not None else sim_i.backend_id
        
        from src.models.tokenization import tokenize as _tok
        toks = _tok(t, sigma, r, freq, be)
        feat_keys = ['t_norm', 'dt_prev', 'r_over_sig', 'log_sigma', 'r_raw', 'freq_norm']
        feats = torch.stack([toks[k] for k in feat_keys], dim=-1).unsqueeze(0)
        batch_m = {
            'features': feats.to(device),
            'backend_id': toks['backend_id'].unsqueeze(0).to(device),
            'mask': torch.ones(1, len(t), dtype=torch.bool).to(device),
            'tspan_yr': torch.tensor([float(t.max() - t.min())]).to(device),
        }
        
        # Exact posterior (full data, raw sigma)
        ex = exact_posterior_grid(
            sim_i.residuals, sim_i.sigma, sim_i.F, sim_i.tspan,
            sim_i.n_modes, prior_bounds, n_grid=n_grid, jitter=jitter)
        
        # Learned global flow on 2D red-noise slice
        with torch.no_grad():
            g_ctx, _ = model._get_contexts(batch_m)
            g_ctx_exp = g_ctx.expand(n_grid * n_grid, -1)
            theta_norm_m = model._normalize_global(grid_4d)
            lp_norm = model.global_flow.log_prob(theta_norm_m.float(), g_ctx_exp.float())
            lp = (lp_norm - model.global_theta_std.log().sum()).cpu().numpy()
        
        lp_2d = lp.reshape(n_grid, n_grid)
        lmax = np.max(lp_2d)
        lnorm = lmax + np.log(np.sum(np.exp(lp_2d - lmax)) * dA * dG)
        lpost = np.exp(lp_2d - lnorm)
        h_list.append(hellinger_distance_grid(ex['posterior'], lpost))
    
    hellinger_by_level[sev] = h_list
    print(f'Severity {sev:.2f}: H = {np.mean(h_list):.4f} ± {np.std(h_list):.4f}')

fig, ax = plt.subplots(figsize=(8, 4.5))
means = [np.mean(hellinger_by_level[s]) for s in masking_levels]
stds = [np.std(hellinger_by_level[s]) for s in masking_levels]
ax.errorbar(masking_levels, means, yerr=stds, fmt='o-', capsize=4, lw=1.5,
            color='steelblue', label='Transformer (global flow)')
ax.set_xlabel('Masking severity')
ax.set_ylabel('Hellinger distance')
ax.set_title('Robustness to Structured Masking', fontsize=12)
ax.legend()
ax.set_xlim(-0.05, 0.65)
fig.tight_layout()
plt.show()

print('\nA robust model shows only modest Hellinger increase with masking.')

---
## 8. Running the Full Evaluation Script

For a comprehensive evaluation from the command line:

```bash
# Transformer only
python -m src.evaluate \
    --config configs/transformer_v5.yaml \
    --checkpoint outputs/v5/transformer/best_model.pt

# Transformer vs LSTM comparison
python -m src.evaluate \
    --config configs/transformer_v5.yaml \
    --checkpoint outputs/v5/transformer/best_model.pt \
    --baseline-checkpoint outputs/v5/lstm/best_model.pt
```

Outputs:
- `outputs/v5/eval/eval_metrics.json` — summary metrics at all masking levels
- `outputs/v5/eval/transformer/posterior_*.png` — posterior comparison plots
- `outputs/v5/eval/transformer/pp.png` — P-P calibration plot
- `outputs/v5/eval/robustness.png` — transformer vs LSTM across masking levels

In [ ]:
# Load saved eval metrics if available
eval_path = os.path.join(PROJECT_ROOT, 'outputs', 'v5', 'eval', 'eval_metrics.json')

if os.path.exists(eval_path):
    import json
    with open(eval_path) as f:
        eval_metrics = json.load(f)
    
    print('Saved evaluation results:\n')
    for model_name, levels in eval_metrics.items():
        print(f'=== {model_name} ===')
        for sev, metrics in levels.items():
            print(f'  Severity {sev}: H={metrics["hellinger"]:.4f}, '
                  f'KS={metrics["ks_mean"]:.4f}, '
                  f'PE={metrics["point_error"]:.4f}')
else:
    print('No saved evaluation metrics found.')
    print('Run the evaluation script to generate them.')

---
## Summary

The v5 evaluation pipeline assesses the factorized posterior across all components:

| Metric | Measures | Good values |
|--------|----------|-----------|
| **Hellinger distance** | Agreement with exact red-noise posterior | < 0.1 |
| **KS statistic** | Calibration across 4 global params | < 0.1 |
| **Point estimate error** | Accuracy of posterior mean (4D global) | As small as possible |
| **Robustness curve** | Degradation under masking | Flat or gentle slope |

### Key takeaways

1. `FactorizedNPEModel.sample_posterior()` returns `(global_samples, wn_samples)` — a tuple
2. The **global flow** produces 4D posteriors over (log10_A_red, γ_red, log10_A_dm, γ_dm)
3. The **WN flow** produces 3D posteriors per backend over (EFAC, log10_EQUAD, log10_ECORR)
4. For comparison with the 2D exact posterior, we evaluate a red-noise slice with DM noise at prior midpoints
5. **P-P calibration** now covers all 4 global parameters — systematic biases are visible per-parameter
6. The evaluation script handles everything end-to-end, including Transformer vs LSTM comparison

This completes the tutorial series. See the [overview tutorial](../tutorial_sbi_framework.ipynb) for a single-notebook walkthrough of the entire pipeline.